# Task 5 - Traffic Recommendation System
## Practical Travel Time Recommendations

This notebook transforms analysis and model outputs into a practical recommendation system that:
- Identifies lower-traffic travel periods
- Considers day type and weather conditions
- Recommends optimal travel windows
- Generates plain-language recommendations

**Example Output:**
"For a weekday journey, consider travelling between 10:00 and 11:00 AM, when historical traffic volumes are typically 35% lower than peak hours."

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('OK - Libraries imported')

OK - Libraries imported


## 1. Load and Analyze Historical Traffic Data

In [2]:
df = pd.read_csv('Metro_Interstate_Traffic_Volume_part3_preprocessed.csv')
df['date_time'] = pd.to_datetime(df['date_time'])
df['hour'] = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print(f'Historical data loaded: {len(df):,} records')
print(f'Date range: {df["date_time"].min()} to {df["date_time"].max()}')
print(f'Mean traffic volume: {df["traffic_volume"].mean():.0f} vehicles/hour')
print(f'Peak traffic: {df["traffic_volume"].max():.0f} vehicles/hour')
print(f'Low traffic: {df["traffic_volume"].min():.0f} vehicles/hour')

Historical data loaded: 48,187 records
Date range: 2012-10-02 09:00:00 to 2018-09-30 23:00:00
Mean traffic volume: 3260 vehicles/hour
Peak traffic: 7280 vehicles/hour
Low traffic: 0 vehicles/hour


## 2. Build Traffic Analysis by Hour

In [3]:
# Analyze traffic patterns by hour
hourly_stats = df.groupby('hour')['traffic_volume'].agg([
    ('mean', 'mean'),
    ('std', 'std'),
    ('min', 'min'),
    ('max', 'max'),
    ('p25', lambda x: x.quantile(0.25)),
    ('p50', lambda x: x.quantile(0.50)),
    ('p75', lambda x: x.quantile(0.75))
])

# Calculate percentile rank for each hour
hourly_stats['traffic_rank'] = hourly_stats['mean'].rank() / len(hourly_stats) * 100

print('Hourly Traffic Statistics:')
print(hourly_stats.round(0))

Hourly Traffic Statistics:
        mean     std  min   max     p25     p50     p75  traffic_rank
hour                                                                 
0      835.0   381.0    6  3075   570.0   676.0  1078.0          21.0
1      516.0   227.0    2  1806   353.0   420.0   673.0          12.0
2      388.0   168.0    3  1432   269.0   315.0   506.0           8.0
3      371.0    67.0    1   930   337.0   362.0   393.0           4.0
4      703.0   234.0    7  1334   440.0   807.0   870.0          17.0
5     2095.0  1008.0  208  3481   780.0  2638.0  2857.0          29.0
6     4141.0  2073.0  241  6386  1401.0  5381.0  5689.0          50.0
7     4740.0  2215.0  190  7260  2054.0  5998.0  6478.0          83.0
8     4587.0  1669.0    1  6888  2976.0  5440.0  5890.0          71.0
9     4385.0  1048.0    1  6063  3648.0  4839.0  5164.0          62.0
10    4184.0   612.0    3  5422  3918.0  4366.0  4584.0          54.0
11    4466.0   619.0   24  6012  4188.0  4576.0  4860.0        

In [4]:
# Analyze by day type (weekday vs weekend)
weekday_stats = df[df['is_weekend'] == 0].groupby('hour')['traffic_volume'].agg(['mean', 'std'])
weekend_stats = df[df['is_weekend'] == 1].groupby('hour')['traffic_volume'].agg(['mean', 'std'])

# Analyze by weather
weather_stats = df.groupby('weather_main')['traffic_volume'].agg(['mean', 'count']).sort_values('mean', ascending=False)

print('\nWeather Impact on Traffic:')
print(weather_stats.round(0))


Weather Impact on Traffic:
                mean  count
weather_main               
Clouds        3618.0  15158
Haze          3502.0   1360
Rain          3318.0   5672
Drizzle       3292.0   1820
Smoke         3238.0     20
Clear         3056.0  13384
Snow          3016.0   2875
Thunderstorm  2999.0   1033
Mist          2933.0   5949
Fog           2704.0    912
Squall        2062.0      4


## 3. Identify Optimal Travel Windows

In [5]:
def identify_optimal_hours(stats, threshold_percentile=30):
    """
    Identify hours with lowest traffic (bottom threshold_percentile)
    """
    threshold = np.percentile(stats['mean'], threshold_percentile)
    optimal_hours = stats[stats['mean'] <= threshold].index.tolist()
    return sorted(optimal_hours)

# Find optimal hours overall and by day type
optimal_hours_overall = identify_optimal_hours(hourly_stats, threshold_percentile=25)
optimal_hours_weekday = identify_optimal_hours(weekday_stats, threshold_percentile=25)
optimal_hours_weekend = identify_optimal_hours(weekend_stats, threshold_percentile=25)

print(f'Optimal travel hours (lowest traffic): {optimal_hours_overall}')
print(f'Optimal weekday hours: {optimal_hours_weekday}')
print(f'Optimal weekend hours: {optimal_hours_weekend}')

Optimal travel hours (lowest traffic): [0, 1, 2, 3, 4, 23]
Optimal weekday hours: [0, 1, 2, 3, 4, 23]
Optimal weekend hours: [1, 2, 3, 4, 5, 6]


In [6]:
def find_continuous_window(hours, window_size=2):
    """
    Find the best continuous window of low-traffic hours
    """
    if len(hours) < window_size:
        return hours[:1] if hours else []
    
    best_window = None
    best_avg = float('inf')
    
    for i in range(len(hours) - window_size + 1):
        window = hours[i:i+window_size]
        if window[-1] - window[0] == window_size - 1:  # continuous
            avg_traffic = hourly_stats.loc[window, 'mean'].mean()
            if avg_traffic < best_avg:
                best_avg = avg_traffic
                best_window = window
    
    if best_window is None and hours:
        best_window = [hours[0]]
    
    return best_window if best_window else []

# Find best travel windows
best_window_overall = find_continuous_window(optimal_hours_overall, window_size=2)
best_window_weekday = find_continuous_window(optimal_hours_weekday, window_size=2)
best_window_weekend = find_continuous_window(optimal_hours_weekend, window_size=2)

print(f'\nBest overall window: {best_window_overall}')
print(f'Best weekday window: {best_window_weekday}')
print(f'Best weekend window: {best_window_weekend}')


Best overall window: [2, 3]
Best weekday window: [2, 3]
Best weekend window: [2, 3]


## 4. Build Recommendation Engine

In [7]:
class TrafficRecommendationEngine:
    def __init__(self, df, hourly_stats):
        self.df = df
        self.hourly_stats = hourly_stats
        self.avg_traffic = df['traffic_volume'].mean()
        self.peak_traffic = df['traffic_volume'].max()
        
    def get_recommendation(self, day_type='weekday', weather_pref=None):
        """
        Generate travel recommendation based on day type and weather
        day_type: 'weekday' or 'weekend'
        weather_pref: 'any' or specific weather type (optional)
        """
        # Filter by day type
        if day_type == 'weekday':
            day_data = self.df[self.df['is_weekend'] == 0]
            day_name = 'weekday'
        else:
            day_data = self.df[self.df['is_weekend'] == 1]
            day_name = 'weekend'
        
        # Filter by weather if specified
        if weather_pref and weather_pref != 'any':
            day_data = day_data[day_data['weather_main'].str.lower() == weather_pref.lower()]
        
        if len(day_data) == 0:
            return f'No data available for {day_name} {weather_pref or ""} conditions.'
        
        # Analyze by hour for this subset
        hourly = day_data.groupby('hour')['traffic_volume'].mean()
        best_hour = hourly.idxmin()
        best_traffic = hourly[best_hour]
        
        # Find continuous low-traffic window
        sorted_hours = hourly.nsmallest(4).index.tolist()
        window_start = min(sorted_hours)
        window_end = window_start + 1
        
        # Calculate improvement
        avg_day_traffic = hourly.mean()
        improvement = ((avg_day_traffic - best_traffic) / avg_day_traffic) * 100
        
        # Generate recommendation
        time_str = f'{window_start:02d}:00 and {window_end:02d}:00'
        weather_str = f' with {weather_pref} weather' if weather_pref and weather_pref != 'any' else ''
        
        recommendation = (
            f'For a {day_name} journey{weather_str}, '
            f'consider travelling between {time_str}, '
            f'when traffic volumes are typically {improvement:.0f}% lower than average.'
        )
        
        stats = {
            'day_type': day_name,
            'recommended_window': f'{window_start:02d}:00-{window_end:02d}:00',
            'traffic_at_window': int(best_traffic),
            'average_traffic': int(avg_day_traffic),
            'improvement_percent': improvement,
            'weather': weather_pref
        }
        
        return recommendation, stats

engine = TrafficRecommendationEngine(df, hourly_stats)
print('Recommendation engine initialized')

Recommendation engine initialized


## 5. Generate Recommendations

In [8]:
# Test recommendations
print('\n' + '='*70)
print('TRAFFIC TRAVEL RECOMMENDATIONS')
print('='*70)

scenarios = [
    ('weekday', None),
    ('weekend', None),
    ('weekday', 'Clear'),
    ('weekday', 'Rainy')
]

recommendations = []
for day_type, weather in scenarios:
    rec, stats = engine.get_recommendation(day_type, weather)
    recommendations.append(stats)
    print(f'\n{rec}')
    print(f'  → Recommended time: {stats["recommended_window"]}')
    print(f'  → Expected traffic: {stats["traffic_at_window"]} vehicles/hour')
    print(f'  → Improvement: {stats["improvement_percent"]:.1f}% below average')

print('\n' + '='*70)


TRAFFIC TRAVEL RECOMMENDATIONS

For a weekday journey, consider travelling between 00:00 and 01:00, when traffic volumes are typically 91% lower than average.
  → Recommended time: 00:00-01:00
  → Expected traffic: 301 vehicles/hour
  → Improvement: 91.5% below average

For a weekend journey, consider travelling between 02:00 and 03:00, when traffic volumes are typically 86% lower than average.
  → Recommended time: 02:00-03:00
  → Expected traffic: 375 vehicles/hour
  → Improvement: 85.6% below average

For a weekday journey with Clear weather, consider travelling between 00:00 and 01:00, when traffic volumes are typically 92% lower than average.
  → Recommended time: 00:00-01:00
  → Expected traffic: 307 vehicles/hour
  → Improvement: 91.5% below average


ValueError: too many values to unpack (expected 2)

## 6. Visualize Traffic Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Hourly traffic pattern
ax = axes[0, 0]
ax.plot(hourly_stats['mean'], marker='o', linewidth=2, markersize=6, color='#3498db')
ax.fill_between(hourly_stats.index, hourly_stats['p25'], hourly_stats['p75'], alpha=0.3)
ax.set_xlabel('Hour of Day', fontsize=11)
ax.set_ylabel('Traffic Volume (vehicles/hour)', fontsize=11)
ax.set_title('Average Hourly Traffic Pattern', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 24, 2))

# Plot 2: Weekday vs Weekend
ax = axes[0, 1]
ax.plot(weekday_stats['mean'], marker='o', label='Weekday', linewidth=2, color='#e74c3c')
ax.plot(weekend_stats['mean'], marker='s', label='Weekend', linewidth=2, color='#2ecc71')
ax.set_xlabel('Hour of Day', fontsize=11)
ax.set_ylabel('Traffic Volume', fontsize=11)
ax.set_title('Weekday vs Weekend Traffic', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 24, 2))

# Plot 3: Weather impact
ax = axes[1, 0]
weather_stats.sort_values('mean', ascending=True)['mean'].plot(kind='barh', ax=ax, color='#9b59b6')
ax.set_xlabel('Average Traffic Volume', fontsize=11)
ax.set_title('Traffic by Weather Condition', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Plot 4: Traffic rank by hour
ax = axes[1, 1]
colors = ['#2ecc71' if r < 30 else '#f39c12' if r < 70 else '#e74c3c' for r in hourly_stats['traffic_rank']]
ax.bar(hourly_stats.index, hourly_stats['traffic_rank'], color=colors, alpha=0.8, edgecolor='black')
ax.set_xlabel('Hour of Day', fontsize=11)
ax.set_ylabel('Traffic Percentile Rank', fontsize=11)
ax.set_title('Traffic Intensity by Hour\n(Green=Low, Yellow=Medium, Red=High)', fontsize=12, fontweight='bold')
ax.axhline(y=30, color='green', linestyle='--', linewidth=1, alpha=0.5, label='Low threshold')
ax.axhline(y=70, color='red', linestyle='--', linewidth=1, alpha=0.5, label='High threshold')
ax.set_xticks(range(0, 24, 2))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('08_traffic_patterns.png', dpi=300, bbox_inches='tight')
plt.show()
print('Traffic pattern visualization saved')

## 7. Key Insights and Patterns

In [ ]:
print('\n' + '='*70)
print('KEY TRAFFIC INSIGHTS')
print('='*70)

print('\n1. PEAK HOURS (Avoid these times):')
peak_hours = hourly_stats.nlargest(3, 'mean')
for hour, row in peak_hours.iterrows():
    print(f'   {hour:02d}:00 - {int(row["mean"])} vehicles/hour')

print('\n2. OPTIMAL TRAVEL HOURS (Recommended):')
optimal = hourly_stats.nsmallest(3, 'mean')
for hour, row in optimal.iterrows():
    improvement = ((hourly_stats['mean'].mean() - row['mean']) / hourly_stats['mean'].mean()) * 100
    print(f'   {hour:02d}:00 - {int(row["mean"])} vehicles/hour ({improvement:.0f}% below average)')

print('\n3. WEEKDAY PATTERNS:')
wd_peak = weekday_stats['mean'].idxmax()
wd_low = weekday_stats['mean'].idxmin()
print(f'   Peak hour: {wd_peak:02d}:00 ({int(weekday_stats.loc[wd_peak, "mean"])} vehicles/hour)')
print(f'   Lowest: {wd_low:02d}:00 ({int(weekday_stats.loc[wd_low, "mean"])} vehicles/hour)')
wd_saving = ((weekday_stats.loc[wd_peak, 'mean'] - weekday_stats.loc[wd_low, 'mean']) / weekday_stats.loc[wd_peak, 'mean']) * 100
print(f'   Potential time saving: {wd_saving:.0f}% traffic reduction')

print('\n4. WEEKEND PATTERNS:')
we_peak = weekend_stats['mean'].idxmax()
we_low = weekend_stats['mean'].idxmin()
print(f'   Peak hour: {we_peak:02d}:00 ({int(weekend_stats.loc[we_peak, "mean"])} vehicles/hour)')
print(f'   Lowest: {we_low:02d}:00 ({int(weekend_stats.loc[we_low, "mean"])} vehicles/hour)')
we_saving = ((weekend_stats.loc[we_peak, 'mean'] - weekend_stats.loc[we_low, 'mean']) / weekend_stats.loc[we_peak, 'mean']) * 100
print(f'   Potential time saving: {we_saving:.0f}% traffic reduction')

print('\n5. WEATHER IMPACT:')
worst_weather = weather_stats['mean'].idxmax()
best_weather = weather_stats['mean'].idxmin()
weather_impact = ((weather_stats.loc[worst_weather, 'mean'] - weather_stats.loc[best_weather, 'mean']) / weather_stats.loc[worst_weather, 'mean']) * 100
print(f'   Worst: {worst_weather} ({int(weather_stats.loc[worst_weather, "mean"])} vehicles/hour)')
print(f'   Best: {best_weather} ({int(weather_stats.loc[best_weather, "mean"])} vehicles/hour)')
print(f'   Weather impact: {weather_impact:.0f}% difference')

## 8. Practical Usage Examples

In [ ]:
print('\n' + '='*70)
print('EXAMPLE USE CASES')
print('='*70)

print('\nUSE CASE 1: Commuter Optimizing Weekday Journey')
rec1, stats1 = engine.get_recommendation('weekday')
print(f'Query: "When should I leave on a typical workday?"')
print(f'Answer: {rec1}')

print('\nUSE CASE 2: Weekend Trip Planner')
rec2, stats2 = engine.get_recommendation('weekend')
print(f'Query: "Best time for a weekend trip?"')
print(f'Answer: {rec2}')

print('\nUSE CASE 3: Weather-Aware Travel')
clear_days = df[df['weather_main'] == 'Clear']
if len(clear_days) > 100:
    rec3, stats3 = engine.get_recommendation('weekday', 'Clear')
    print(f'Query: "Best time to travel on a clear weekday?"')
    print(f'Answer: {rec3}')

print('\nUSE CASE 4: Business Decision')
print('Query: "Which day/time minimizes travel time?"')
print(f'Answer: {best_window_overall}, with average traffic of {int(hourly_stats.loc[best_window_overall[0], "mean"])} vehicles/hour')

## 9. System Summary

In [ ]:
print('\n' + '='*70)
print('TRAFFIC RECOMMENDATION SYSTEM - SUMMARY')
print('='*70)

print('\n1. SYSTEM CAPABILITIES:')
print('   ✓ Analyzes 48,000+ historical traffic records')
print('   ✓ Identifies optimal travel windows for any day/weather')
print('   ✓ Generates personalized, plain-language recommendations')
print('   ✓ Quantifies potential time savings')

print('\n2. DATA-DRIVEN INSIGHTS:')
print(f'   • Peak traffic: {df["traffic_volume"].max():.0f} vehicles/hour')
print(f'   • Off-peak traffic: {df["traffic_volume"].min():.0f} vehicles/hour')
print(f'   • Average traffic: {df["traffic_volume"].mean():.0f} vehicles/hour')
print(f'   • Optimal hours: {len(optimal_hours_overall)} hours per day with low traffic')

print('\n3. RECOMMENDATION FACTORS:')
print('   • Time of day (hour-based analysis)')
print('   • Day type (weekday vs weekend)')
print('   • Weather conditions')
print('   • Historical traffic patterns')

print('\n4. OUTPUT QUALITY:')
print('   • Human-readable recommendations with context')
print('   • Quantified benefits (e.g., "35% lower traffic")')
print('   • Statistical backing from historical analysis')
print('   • Actionable time windows (specific hours)')

print('\n5. POTENTIAL APPLICATIONS:')
print('   • Mobile app for travelers')
print('   • Fleet optimization systems')
print('   • Logistics route planning')
print('   • Traffic management dashboards')
print('   • Ride-sharing demand prediction')

print('\n' + '='*70)